# EfficientAD-S MVTec AD training on Kaggle

This notebook clones the project branch, installs the pinned Kaggle experiment dependencies, validates the attached MVTec AD dataset, and trains category-specific EfficientAD-S models. It is designed to be uploaded and run as one notebook.

MVTec AD remains subject to CC BY-NC-SA 4.0. Keep the attached dataset and generated checkpoints private unless the license and intended use permit distribution.

In [ ]:
# Kaggle setup: enable Internet in Notebook Settings before running this cell.
from pathlib import Path
import json
import os
import subprocess
import sys
import time

REPO_URL = 'https://github.com/Deepak-kumar-36/Agentic-AI-Powered-Smart-Manufacturing-Quality-Inspection-and-Automation.git'
REPO_BRANCH = 'deepak/feat/local-inspection-api'
REPO_DIR = Path('/kaggle/working/quality-inspection')
DATASET_ROOT = Path('/kaggle/input/datasets/ipythonx/mvtec-ad')
ARTIFACT_ROOT = Path('/kaggle/working/efficientad_s_artifacts')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

sys.path.insert(0, str(REPO_DIR))
COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Repository:', REPO_DIR)
print('Branch:', REPO_BRANCH)
print('Commit:', COMMIT)
print('Dataset:', DATASET_ROOT)

In [ ]:
# Install only the Kaggle experiment dependencies.
# Use explicit arguments here so an uploaded notebook remains correct even if the remote branch has not refreshed its requirements file yet.
# Kaggle's current Python 3.13 image requires the torch 2.6/torchvision 0.21 pair.
kaggle_packages = [
    'anomalib==2.2.0', 'torch==2.6.0', 'torchvision==0.21.0',
    'lightning==2.5.0.post0', 'numpy==2.1.3', 'Pillow==11.1.0',
    'scipy==1.15.1', 'scikit-learn==1.6.1',
    'transformers==4.49.0', 'huggingface-hub==0.30.2', 'tokenizers==0.21.1',
    'requests==2.32.4', 'packaging==25.0',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', *kaggle_packages], check=True)
# Verify the package versions that matter to the Anomalib import path.
import importlib.metadata as metadata
for package_name in ['torch', 'torchvision', 'numpy', 'scipy', 'anomalib', 'lightning', 'torchmetrics', 'transformers', 'huggingface-hub', 'requests']:
    try: print(package_name, metadata.version(package_name))
    except metadata.PackageNotFoundError: print(package_name, 'MISSING')
print('Dependencies installed. Restart the Kaggle session now, then rerun the notebook from the first cell so NumPy/SciPy and package metadata are loaded coherently.')

import torch
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA version:', torch.version.cuda)
if not torch.cuda.is_available():
    print('WARNING: GPU is unavailable. Training will be much slower on CPU.')
else:
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Run the repository's offline tests before training.
test_env = os.environ.copy()
test_env['PYTHONPATH'] = str(REPO_DIR) + os.pathsep + test_env.get('PYTHONPATH', '')
test_result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(REPO_DIR / 'tests'), '-v'], cwd=str(REPO_DIR), env=test_env, text=True)
if test_result.returncode != 0:
    raise RuntimeError('Repository tests failed; stop before training.')
print('Repository tests passed.')

## Dataset validation

The dataset path is intentionally fixed to `/kaggle/input/datasets/ipythonx/mvtec-ad`. The reusable project manifest validates image readability, category structure, defect masks, deterministic splits, and file hashes before training.

In [ ]:
from app.efficientad.data import discover_mvtec_manifest

if not DATASET_ROOT.is_dir():
    raise RuntimeError(f'MVTec AD dataset was not found at {DATASET_ROOT}')
if not (DATASET_ROOT / 'bottle' / 'train' / 'good').is_dir():
    raise RuntimeError(f'Invalid MVTec AD root; expected category directories below {DATASET_ROOT}')

CATEGORIES = ['bottle', 'screw', 'leather']
manifest = discover_mvtec_manifest(DATASET_ROOT, CATEGORIES)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
manifest_path = ARTIFACT_ROOT / 'dataset_manifest.json'
if manifest_path.exists():
    existing_manifest = manifest_path.read_text(encoding='utf-8')
    if existing_manifest != manifest.stable_json() + '\n':
        raise RuntimeError(f'Existing manifest differs from the current dataset: {manifest_path}')
    print('Reusing verified existing manifest:', manifest_path)
else:
    manifest.write(manifest_path)

print('Categories:', manifest.categories)
print('Rows:', len(manifest.rows))
print('Manifest SHA-256:', manifest.digest())
for category in manifest.categories:
    rows = [row for row in manifest.rows if row.category == category]
    print(category, {'train': sum(row.split == 'train' for row in rows), 'validation': sum(row.split == 'validation' for row in rows), 'test': sum(row.split == 'test' for row in rows), 'anomaly_test': sum(row.split == 'test' and row.label == 1 for row in rows)})

## Experiment configuration and provenance

EfficientAD-S is provided through Anomalib 2.2.0. The teacher weights are acquired by the pinned library during model initialization. A failed acquisition must stop the run; this notebook never substitutes random weights.

In [ ]:
from app.efficientad.config import EfficientADConfig

config = EfficientADConfig(
    model_size='small',
    image_size=(256, 256),
    batch_size=32,
    epochs=100,
    learning_rate=1e-4,
    seed=42,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
print(json.dumps(config.as_dict(), indent=2))
print('Config SHA-256:', config.digest())

In [ ]:
# Deterministic seeds for the experiment envelope.
import random
import numpy as np
random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(config.seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print('Seed configured:', config.seed)

## Category-specific EfficientAD-S training

This cell trains one model per category using Anomalib's MVTec datamodule and `EfficientAd(model_size='small')`. Final test data is reserved for the library's test phase; threshold tuning must be performed from validation outputs before claiming qualification.

In [ ]:
from anomalib.data import MVTec
from anomalib.engine import Engine
from anomalib.models import EfficientAd

training_summary = []
for category in CATEGORIES:
    category_root = ARTIFACT_ROOT / category
    category_root.mkdir(parents=True, exist_ok=True)
    started = time.time()
    print(f'\n=== Training {category} ===')
    datamodule = MVTec(
        root=str(DATASET_ROOT),
        category=category,
        image_size=config.image_size,
        train_batch_size=config.batch_size,
        eval_batch_size=config.batch_size,
        num_workers=config.num_workers,
    )
    model = EfficientAd(model_size='small')
    engine = Engine(default_root_dir=str(category_root), max_epochs=config.epochs)
    engine.fit(model=model, datamodule=datamodule)
    test_result = engine.test(model=model, datamodule=datamodule)
    duration = time.time() - started
    training_summary.append({'category': category, 'duration_seconds': duration, 'test_result': str(test_result), 'checkpoint_directory': str(category_root)})
    print('Finished', category, 'in', round(duration, 2), 'seconds')

print(json.dumps(training_summary, indent=2))

## Export run metadata

This metadata is deliberately conservative. It records the run environment and hashes the dataset/configuration. Replace the qualification status only after reviewing actual validation thresholds, untouched test metrics, checkpoint hashes, and artifact completeness.

In [ ]:
run_metadata = {
    'experiment': 'efficientad-s-mvtec-v1',
    'repository_url': REPO_URL,
    'repository_branch': REPO_BRANCH,
    'repository_commit': COMMIT,
    'dataset_root': str(DATASET_ROOT),
    'dataset_manifest_sha256': manifest.digest(),
    'config': config.as_dict(),
    'config_sha256': config.digest(),
    'categories': CATEGORIES,
    'license': 'CC BY-NC-SA 4.0',
    'python': sys.version,
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'training_summary': training_summary,
    'qualification_status': 'NOT_QUALIFIED_UNTIL_METRICS_AND_ARTIFACT_HASHES_ARE_REVIEWED'
}
metadata_path = ARTIFACT_ROOT / 'run_metadata.json'
metadata_path.write_text(json.dumps(run_metadata, indent=2, default=str) + '\n', encoding='utf-8')
print(json.dumps(run_metadata, indent=2, default=str))
print('Artifacts:', ARTIFACT_ROOT)

## Download outputs

After the run completes, use Kaggle's file browser to download `/kaggle/working/efficientad_s_artifacts/`. Place the downloaded category artifacts under the local project's configured model-artifact directory only after verifying the manifest and hashes. Do not commit checkpoints, datasets, or generated outputs to GitHub.